In [1]:
! pip install requests bs4 agent-framework-core -U


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework import WorkflowEvent, WorkflowBuilder, WorkflowRunResult
from OnetWebService import OnetWebService
import sys
import os
import requests
import re
from bs4 import BeautifulSoup
from dotenv import load_dotenv
load_dotenv()

chat_client = OpenAIChatCompletionClient(
    base_url=os.environ.get("GITHUB_ENDPOINT"),    # GitHub Models API endpoint
    api_key=os.environ.get("GITHUB_TOKEN"),        # Authentication token
    model=os.environ.get("GITHUB_MODEL_ID")  # Model for all agents in workflow
)

In [3]:
import urllib.request, urllib.parse, urllib.error
import json
import os
from dotenv import load_dotenv

class OnetWebService:
    def __init__(self, api_key):
        self._headers = {
            'User-Agent': 'python-OnetWebService/2.00 (bot)',
            'X-API-Key': api_key,
            'Accept': 'application/json' }
        self.set_version()
    
    def set_version(self, version = None):
        if version is None:
            self._url_root = 'https://api-v2.onetcenter.org/'
        else:
            self._url_root = 'https://api-v' + version + '.onetcenter.org/'
    
    def call(self, path, *query):
        try:
            url = self._url_root + path
            if len(query) > 0:
                url += '?' + urllib.parse.urlencode(query, True)
            req = urllib.request.Request(url, None, self._headers)
            handle = None
            try:
                handle = urllib.request.urlopen(req)
            except urllib.error.HTTPError as e:
                if e.code == 422:
                    try:
                        return json.load(e)
                    except json.JSONDecodeError:
                        return { 'error': 'Call to ' + url + ' failed to return valid JSON' }
                    except UnicodeDecodeError:
                        return { 'error': 'Call to ' + url + ' failed to return valid UTF-8' }
                else:
                    return { 'error': 'Call to ' + url + ' failed with error code ' + str(e.code) }
            except urllib.error.URLError as e:
                return { 'error': 'Call to ' + url + ' failed with reason: ' + str(e.reason) }
            code = handle.getcode()
            if (code != 200) and (code != 422):
                return { 'error': 'Call to ' + url + ' failed with error code ' + str(code),
                        'urllib2_info': handle }
            try:
                return json.load(handle)
            except json.JSONDecodeError:
                return { 'error': 'Call to ' + url + ' failed to return valid JSON' }
            except UnicodeDecodeError:
                return { 'error': 'Call to ' + url + ' failed to return valid UTF-8' }
        except Exception as e:
            return { 'error': 'Call failed with unexpected error', 'exception': e }

In [4]:
import traceback

def get_onet_skills(job_title: str) -> str:
    try:
        api_key = os.environ.get("ONET_API_KEY")
        onet_ws = OnetWebService(api_key)

        kwresults = onet_ws.call('online/search', ('keyword', job_title), ('end', 1))
        if 'error' in kwresults:
            return f"O*NET API Error: {kwresults['error']}"
        if (not 'occupation' in kwresults) or (0 == len(kwresults['occupation'])):
            return f"No relevant occupations were found for {job_title}."

        soc_code = kwresults['occupation'][0]['code']

        endpoints_to_check = [
            f'online/occupations/{soc_code}/summary/skills',
            f'online/occupations/{soc_code}/summary/knowledge'
        ]

        extracted_items = []
        
        def extract_names(data):
            if isinstance(data, dict):
                for key, value in data.items():
                    if key in ['name', 'title'] and isinstance(value, str):
                        extracted_items.append(value)
                    else:
                        extract_names(value)
            elif isinstance(data, list):
                for item in data:
                    extract_names(item)

        for endpoint in endpoints_to_check:
            res = onet_ws.call(endpoint)
            if 'error' not in res:
                extract_names(res)

        unique_items = list(set([s for s in extracted_items if len(s) > 2]))
        
        if not unique_items:
            return f"No core skills/knowledge found for {soc_code}."

        return "O*NET Core Knowledge & Skills: " + ", ".join(unique_items[:15])

    except Exception as e:
        return f"PYTHON CRASH TRACE: {traceback.format_exc()}"

In [5]:
def search_soc(keyword: str, subject_code: str) -> str:
    url = "https://sis.rutgers.edu/soc/api/courses.json"
    params = {
        "year": "2026",
        "term": "9",
        "campus": "NB",
        "level": "U",
        "subject": subject_code
    }

    blacklist = ["money", "liberal arts", "business of", "for non-science", "elementary"]

    try:
        response = requests.get(url, params=params, timeout=5)
        response.raise_for_status()
        courses = response.json()
        
        results = []
        search_terms = [term.lower() for term in keyword.split() if len(term) > 3]
        if not search_terms:
            search_terms = [keyword.lower()]

        for course in courses:
            title = course.get("title", "").lower()
            
            if any(word in title for word in blacklist):
                continue
                
            description = str(course.get("courseDescription", "")).lower()
            expanded_title = str(course.get("expandedTitle", "")).lower()
            search_body = title + " " + description + " " + expanded_title

            if any(term in search_body for term in search_terms):
                course_number = course.get('courseNumber')
                subject = course.get('subject')
                school = course.get('offeringUnitCode', '01')
                
                if subject == "198":
                    clean_title = re.sub(r'[^a-zA-Z0-9\s]', '', title)
                    slug = clean_title.replace(" ", "-").replace("--", "-")
                    synopsis_link = f"https://www.cs.rutgers.edu/academics/undergraduate/course-synopses/course-details/{school}-{subject}-{course_number}-{slug}"
                else:
                    synopsis_link = course.get('synopsisUrl', "No specific link provided")

                course_info = f"Course: {course.get('courseString')} - {course.get('title')}"
                course_info += f"\n   Syllabus/Synopsis Link: {synopsis_link}"
                
                results.append(course_info)
        
        if not results:
            return f"No courses found for keyword: '{keyword}' in subject: {subject_code}"
        
        return f"Courses found for '{keyword}':\n" + "\n\n".join(results[:5])
        
    except requests.exceptions.RequestException as e:
        return f"Tool Error: {str(e)}"

In [6]:
career_agent = chat_client.as_agent(
    name = "Career_Advisor",
    instructions = """You are a strict data-retrieval career advisor. You help people find out what skills they need to develop for their desired career.
    
    CRITICAL RULES:
    1. You MUST use the `get_onet_skills` tool to fetch skills.
    2. If the tool returns an error, fails, or finds no skills, you MUST output the exact error message and STOP. 
    3. NEVER generate, guess, or provide a "general list" of skills from your internal knowledge. 
    4. If you successfully get skills from the tool, summarize them and pass them along with the career title to the Course Advisor.
    """,
    tools = [get_onet_skills]
)

In [7]:
course_agent = chat_client.as_agent(
    name = "course_advisor",
    instructions = """You are an academic routing agent. You receive a list of industry skills and a career title from the Career Advisor.
    
    SEARCH RULES:
    1. NEVER search long phrases like "Database Management Systems". 
    2. You MUST extract a SINGLE word (e.g., "database", "software", "python", "data", "agile") to use as your keyword in the `search_soc` tool.

    Step 1: Map each skill to the most relevant academic departments. Here is your Rutgers STEM subject code reference map:
    -- Sciences & Mathematics (School 01) --
    - 198: Computer Science
    - 640: Mathematics
    - 960: Statistics
    - 119: Biological Sciences
    - 160: Chemistry
    - 750: Physics
    -- USE SOLELY FOR Engineering (School 14) --
    - 540: Industrial Engineering
    - 650: Mechanical Engineering
    - 332: Electrical & Computer Engineering
    - 125: Biomedical Engineering
    - 155: Chemical & Biochemical Engineering
    - 180: Civil & Environmental Engineering
    -- Technology & Informatics --
    - 547: Information Technology & Informatics (ITI)
    
    Step 2: Use the `search_soc` tool to search the catalog. If a skill is broad (like 'Data Analysis' or 'Cybersecurity'), 
    you should call the tool MULTIPLE TIMES using different subject codes (e.g., search once in 198, and once in 245) to cast a wide net.
    
    Step 3: Keep an internal tally of the courses you find and which specific O*NET skills triggered them.
    
    Step 4: Rank the courses. Sort your final output so that courses that cover multiple O*NET skills are at the very top. Format the output cleanly, listing the Course Code, Title, and the specific skills it addresses.
    """,
    tools = [search_soc]
)

In [8]:
career_workflow = WorkflowBuilder(start_executor=career_agent).add_edge(career_agent, course_agent).build()
with open("software_engineer_courses.txt", "w", encoding="utf-8") as f:
    
    print("Starting agent workflow...\n")
    
    async for event in career_workflow.run("What courses should I take to become a software engineer?", stream=True):
        if isinstance(event.data, str):
            continue
        if type(event.data).__name__ == "AgentResponseUpdate":
            chunk = getattr(event.data, "delta", None) or getattr(event.data, "content", None) or getattr(event.data, "text", "")
            if chunk:
                print(chunk, end="", flush=True)
                f.write(chunk)
                f.flush()

Starting agent workflow...

I found the required skills for a Software Engineer. Here they are summarized:

1. **English Language**
2. **Reading Comprehension**
3. **Judgment and Decision Making**
4. **Critical Thinking**
5. **Customer and Personal Service**
6. **Active Learning**
7. **Computers and Electronics**
8. **Programming**
9. **Mathematics**

I'll pass this information along with the career title to the Course Advisor.Here are the courses that would be beneficial for a Software Engineer based on the skills identified:

### Programming Courses
1. **[01:198:214 - SYSTEMS PROGRAMMING](https://www.cs.rutgers.edu/academics/undergraduate/course-synopses/course-details/01-198-214-systems-programming)**
   - Skills: Programming

2. **[01:198:314 - PRIN PROG LANGUAGES](https://www.cs.rutgers.edu/academics/undergraduate/course-synopses/course-details/01-198-314-prin-prog-languages)**
   - Skills: Programming

3. **[04:547:202 - OBJECT-ORIENTED PROG](https://comminfo.rutgers.edu/academic

In [9]:
# workflow = WorkflowBuilder(start_executor=orchestrator_agent).add_edge(career_agent, course_analyzer_agent).build()